# Porosity Control Demo

This tutorial illustrates the hard porosity-control step used after probabilistic generation. It compares quantile-based binarization against a fixed threshold on a lightweight synthetic probability field.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "tutorials":
    ROOT = ROOT.parents[1]
elif ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt

from src.sampling.quantile_binarization import quantile_binarize
from src.metrics.porosity import compute_porosity

In [ ]:
prob_path = ROOT / "examples" / "synthetic_64.npy"
if prob_path.exists():
    prob = np.load(prob_path).astype(np.float32)
else:
    rng = np.random.default_rng(42)
    prob = rng.random((64, 64, 64)).astype(np.float32)

targets = np.array([0.08, 0.10, 0.12, 0.14, 0.16, 0.18])
rows = []
fixed_seg = (prob > 0.5).astype(np.uint8)
fixed_phi = compute_porosity(fixed_seg)

for phi in targets:
    seg, threshold, achieved = quantile_binarize(prob, float(phi), seed=0)
    rows.append((phi, achieved, abs(achieved - phi), threshold, fixed_phi, abs(fixed_phi - phi)))

rows[:3]

In [ ]:
arr = np.array(rows, dtype=float)
plt.figure(figsize=(5, 3.5))
plt.plot(arr[:, 0], arr[:, 2], marker="o", label="quantile projection")
plt.plot(arr[:, 0], arr[:, 5], marker="s", label="fixed threshold")
plt.xlabel("Target porosity")
plt.ylabel("Absolute porosity error")
plt.grid(alpha=0.25)
plt.legend(frameon=False)
plt.tight_layout()

In [ ]:
target = 0.14
seg, threshold, achieved = quantile_binarize(prob, target, seed=0)
mid = prob.shape[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(prob[:, :, mid], cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Probability field")
axes[0].axis("off")
axes[1].imshow(seg[:, :, mid], cmap="gray", vmin=0, vmax=1)
axes[1].set_title(f"Quantile binary, phi={achieved:.3f}")
axes[1].axis("off")
plt.tight_layout()